# SpannerVectorDBStorage + SpannerGraphStorage Integration Test

This notebook tests `SpannerVectorDBStorage` (vector search) and `SpannerGraphStorage` (graph traversal) together, culminating in a full PathRAG end-to-end integration.

## Prerequisites

1. **Google Cloud authentication**
   ```bash
   gcloud auth application-default login
   ```

2. **Enable required APIs**
   ```bash
   gcloud services enable spanner.googleapis.com
   ```

3. **Create a Spanner instance and database** (if not already created)
   ```bash
   export SPANNER_INSTANCE="pathrag"
   export SPANNER_DATABASE="pathrag_db"
   export SPANNER_REGION="us-central1"

   gcloud spanner instances create $SPANNER_INSTANCE \
     --config=regional-$SPANNER_REGION \
     --description="Spanner instance for PathRAG" \
     --nodes=1 \
     --edition=ENTERPRISE

   gcloud spanner databases create $SPANNER_DATABASE \
     --instance=$SPANNER_INSTANCE
   ```

4. **Install dependencies**
   ```bash
   pip install -e .
   pip install google-cloud-spanner python-dotenv
   ```

5. **Configure `.env` file** in `examples/spanner/` (see `.env.example`)

> **Note:** Both `SpannerVectorDBStorage` and `SpannerGraphStorage` automatically create the required tables on first initialization.

---
## Setup: Load environment

In [ ]:
import json
import os

import numpy as np
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

GOOGLE_CLOUD_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT")
SPANNER_INSTANCE = os.environ.get("SPANNER_INSTANCE")
SPANNER_DATABASE = os.environ.get("SPANNER_DATABASE")

print(f"Project:  {GOOGLE_CLOUD_PROJECT}")
print(f"Instance: {SPANNER_INSTANCE}")
print(f"Database: {SPANNER_DATABASE}")

assert SPANNER_INSTANCE and SPANNER_DATABASE, (
    "SPANNER_INSTANCE and SPANNER_DATABASE must be set in .env or environment variables.\n"
    "See examples/spanner/.env.example"
)

### Create mock embedding function and vector storage

Steps 1-4 use a **mock embedding function** (deterministic, no LLM API needed) to test `SpannerVectorDBStorage` in isolation.

In [ ]:
from PathRAG.utils import EmbeddingFunc
from PathRAG.storage.spanner import SpannerVectorDBStorage

MOCK_DIM = 128
VEC_NAMESPACE = "test_vec"

async def mock_embed(texts: list[str]) -> np.ndarray:
    """Deterministic mock embedding: same text always produces the same vector."""
    embeddings = []
    for text in texts:
        seed = sum(ord(c) for c in text) % (2**31)
        rng = np.random.RandomState(seed)
        vec = rng.randn(MOCK_DIM).astype(np.float64)
        vec = vec / np.linalg.norm(vec)
        embeddings.append(vec)
    return np.array(embeddings)

mock_embedding_func = EmbeddingFunc(
    embedding_dim=MOCK_DIM, max_token_size=8192, func=mock_embed
)

vdb = SpannerVectorDBStorage(
    namespace=VEC_NAMESPACE,
    global_config={
        "spanner_project_id": GOOGLE_CLOUD_PROJECT,
        "spanner_instance_id": SPANNER_INSTANCE,
        "spanner_database_id": SPANNER_DATABASE,
        "embedding_batch_num": 32,
    },
    embedding_func=mock_embedding_func,
    meta_fields={"entity_name"},
)
print(f"SpannerVectorDBStorage created (table={vdb._table_name})")

---
## Step 1: Vector Upsert

Insert 5 test vectors with embeddings and metadata, then verify the row count in Spanner.

In [ ]:
test_data = {
    "ent-001": {
        "content": "Apple Inc. is a technology company headquartered in Cupertino.",
        "entity_name": "APPLE",
    },
    "ent-002": {
        "content": "Google LLC is a search engine and cloud computing company.",
        "entity_name": "GOOGLE",
    },
    "ent-003": {
        "content": "Microsoft Corporation develops software and cloud services.",
        "entity_name": "MICROSOFT",
    },
    "ent-004": {
        "content": "Steve Jobs co-founded Apple and was its visionary CEO.",
        "entity_name": "STEVE JOBS",
    },
    "ent-005": {
        "content": "Tim Cook is the current CEO of Apple Inc since 2011.",
        "entity_name": "TIM COOK",
    },
}

print(f"Inserting {len(test_data)} vectors...")
result = await vdb.upsert(test_data)
print(f"Inserted {len(result)} vectors")

# Verify row count
with vdb._database.snapshot() as snap:
    rows = list(snap.execute_sql(f"SELECT COUNT(*) FROM {vdb._table_name}"))
    count = rows[0][0]
print(f"Total rows in '{vdb._table_name}': {count}")
assert count == 5, f"Expected 5 rows, got {count}"
print("PASSED")

---
## Step 2: Vector Similarity Query

Search for similar vectors using `COSINE_DISTANCE` on Spanner.

In [ ]:
# Query 1: technology company
print("Query: 'Apple technology company'")
print("-" * 50)
results = await vdb.query("Apple technology company", top_k=3)
for r in results:
    print(f"  entity={r.get('entity_name', 'N/A'):15s}  similarity={r['distance']:.4f}")
    print(f"    {r['content'][:80]}")
print()

# Query 2: CEO
print("Query: 'CEO of a technology company'")
print("-" * 50)
results = await vdb.query("CEO of a technology company", top_k=3)
for r in results:
    print(f"  entity={r.get('entity_name', 'N/A'):15s}  similarity={r['distance']:.4f}")
    print(f"    {r['content'][:80]}")

---
## Step 3: Upsert Update

Verify that upserting an existing ID overwrites the content and embedding without increasing the row count.

In [ ]:
from google.cloud.spanner_v1 import param_types

# Update APPLE with new content
update_data = {
    "ent-001": {
        "content": "Apple Inc. designs iPhones, iPads, Macs and provides digital services worldwide.",
        "entity_name": "APPLE",
    },
}

print("Updating ent-001 (APPLE) with new content...")
await vdb.upsert(update_data)

# Verify content was updated
with vdb._database.snapshot() as snap:
    rows = list(snap.execute_sql(
        f"SELECT content FROM {vdb._table_name} WHERE id = @id",
        params={"id": "ent-001"},
        param_types={"id": param_types.STRING},
    ))
updated_content = rows[0][0]
print(f"Updated content: {updated_content}")
assert "iPhones" in updated_content, "Content was not updated!"

# Verify row count unchanged
with vdb._database.snapshot() as snap:
    rows = list(snap.execute_sql(f"SELECT COUNT(*) FROM {vdb._table_name}"))
print(f"Total rows (should still be 5): {rows[0][0]}")
assert rows[0][0] == 5
print("PASSED")

---
## Step 4: Delete

Delete a vector row and verify the row count decreases.

In [ ]:
from google.cloud import spanner as spanner_lib

with vdb._database.snapshot() as snap:
    rows = list(snap.execute_sql(f"SELECT COUNT(*) FROM {vdb._table_name}"))
print(f"Rows before deletion: {rows[0][0]}")

# Delete MICROSOFT (ent-003)
print("Deleting ent-003 (MICROSOFT)...")
with vdb._database.batch() as batch:
    batch.delete(
        table=vdb._table_name,
        keyset=spanner_lib.KeySet(keys=[["ent-003"]]),
    )

with vdb._database.snapshot() as snap:
    rows = list(snap.execute_sql(f"SELECT COUNT(*) FROM {vdb._table_name}"))
count_after = rows[0][0]
print(f"Rows after deletion: {count_after}")
assert count_after == 4, f"Expected 4 rows, got {count_after}"
print("PASSED")

---
## Step 5: Full PathRAG Integration (SpannerGraphStorage + SpannerVectorDBStorage)

End-to-end test: index documents and query using PathRAG with **both** Spanner graph and vector storage.

> **Requires:** `GEMINI_API_KEY` or `OPENAI_API_KEY` in `.env`

### 5-1. Configure LLM and create PathRAG instance

In [ ]:
def get_llm_config():
    """Resolve LLM configuration from environment variables."""
    if os.environ.get("GEMINI_API_KEY"):
        default_llm = "gemini/gemini-2.5-flash"
        default_embed = "gemini/gemini-embedding-001"
        default_dim = 3072
    elif os.environ.get("OPENAI_API_KEY"):
        default_llm = "gpt-4o-mini"
        default_embed = "text-embedding-3-small"
        default_dim = 1536
    else:
        raise RuntimeError("No API key found. Set GEMINI_API_KEY or OPENAI_API_KEY.")

    llm_model = os.environ.get("LLM_MODEL_NAME", default_llm)
    embedding_model = os.environ.get("EMBEDDING_MODEL_NAME", default_embed)
    embedding_dim = int(os.environ.get("EMBEDDING_DIM", default_dim))
    tiktoken_model = llm_model

    print(f"LLM Model:       {llm_model}")
    print(f"Embedding Model: {embedding_model} (dim={embedding_dim})")
    return llm_model, embedding_model, embedding_dim, tiktoken_model

llm_model, embedding_model, embedding_dim, tiktoken_model = get_llm_config()

In [ ]:
import tempfile
from PathRAG import PathRAG, QueryParam

work_dir = tempfile.mkdtemp(prefix="pathrag_spanner_nb_")
print(f"Working dir: {work_dir}")

rag = PathRAG(
    working_dir=work_dir,
    llm_model_name=llm_model,
    embedding_model_name=embedding_model,
    embedding_dim=embedding_dim,
    tiktoken_model_name=tiktoken_model,
    graph_storage="SpannerGraphStorage",
    vector_storage="SpannerVectorDBStorage",
    addon_params={
        "spanner_project_id": GOOGLE_CLOUD_PROJECT,
        "spanner_instance_id": SPANNER_INSTANCE,
        "spanner_database_id": SPANNER_DATABASE,
    },
    chunk_token_size=200,
    chunk_overlap_token_size=50,
)

print(f"\nGraph storage:  {type(rag.chunk_entity_relation_graph).__name__}")
print(f"Entities VDB:   {type(rag.entities_vdb).__name__}")
print(f"Relations VDB:  {type(rag.relationships_vdb).__name__}")
print(f"Chunks VDB:     {type(rag.chunks_vdb).__name__}")
print("\nPathRAG instance created with full Spanner backend!")

### 5-2. Index sample documents

In [ ]:
sample_docs = [
    """
    Apple Inc. is an American multinational technology company
    headquartered in Cupertino, California. Apple was founded on
    April 1, 1976, by Steve Jobs, Steve Wozniak, and Ronald Wayne.
    Tim Cook has been the CEO of Apple since August 2011.
    """,
    """
    Google LLC is an American multinational corporation focusing on
    search engine technology and cloud computing. Sundar Pichai has
    been the CEO of Google since October 2015. Google was founded
    on September 4, 1998, by Larry Page and Sergey Brin.
    """,
]

print(f"Indexing {len(sample_docs)} documents...")
await rag.ainsert(sample_docs)
print("Indexing complete!")

### 5-3. Inspect extracted entities and relationships (Spanner Graph)

In [ ]:
graph = rag.chunk_entity_relation_graph
nodes = await graph.nodes()
edges_list = await graph.edges()

print(f"Extracted entities:      {len(nodes)}")
print(f"Extracted relationships: {len(edges_list)}")

print("\n--- Entities (Spanner Graph) ---")
for name in list(nodes)[:10]:
    data = await graph.get_node(name)
    if data:
        print(f"  {name}: type={data.get('entity_type', 'N/A')}")

print("\n--- Relationships (Spanner Graph) ---")
for src, tgt in list(edges_list)[:10]:
    data = await graph.get_edge(src, tgt)
    if data:
        print(f"  {src} --[{data.get('keywords', 'N/A')}]--> {tgt}")

### 5-4. Vector search on entities and relationships (Spanner Vector)

In [ ]:
# Entity vector search
print("Entity Vector Search: 'Apple technology company'")
print("-" * 55)
entity_results = await rag.entities_vdb.query("Apple technology company", top_k=5)
for r in entity_results:
    print(f"  entity={r.get('entity_name', 'N/A'):20s}  similarity={r['distance']:.4f}")

print()

# Relationship vector search
print("Relationship Vector Search: 'founded the company'")
print("-" * 55)
rel_results = await rag.relationships_vdb.query("founded the company", top_k=5)
for r in rel_results:
    print(f"  {r.get('src_id', 'N/A')} -> {r.get('tgt_id', 'N/A')}  similarity={r['distance']:.4f}")

### 5-5. Full RAG queries

In [ ]:
queries = [
    "What is the relationship between Steve Jobs and Apple?",
    "Who founded Google and what is their background?",
    "Compare the leadership of Apple and Google.",
]

for q in queries:
    print(f"Q: {q}")
    response = await rag.aquery(q, param=QueryParam(mode="hybrid", top_k=10))
    display = response[:400] + "..." if len(response) > 400 else response
    print(f"A: {display}")
    print("-" * 70)
    print()

---
## Step 6: Cleanup Test Data

Drop all test tables and the property graph from Spanner.

> **Warning:** This will permanently delete all test data. Only run this when you are done testing.

In [ ]:
from google.cloud import spanner

client = spanner.Client(
    project=GOOGLE_CLOUD_PROJECT,
    disable_builtin_metrics=True,
)
instance = client.instance(SPANNER_INSTANCE)
database = instance.database(SPANNER_DATABASE)

# --- Drop property graph (from PathRAG integration) ---
graph_name = "pathrag_chunk_entity_relation"
print(f"Dropping property graph '{graph_name}'...")
try:
    op = database.update_ddl([f"DROP PROPERTY GRAPH IF EXISTS {graph_name}"])
    op.result()
    print("  Done.")
except Exception as e:
    print(f"  Skipped: {e}")

# --- Drop graph tables (edge first due to FK) ---
for table in ["chunk_entity_relation_edges", "chunk_entity_relation_nodes"]:
    print(f"Dropping table '{table}'...")
    try:
        op = database.update_ddl([f"DROP TABLE IF EXISTS {table}"])
        op.result()
        print("  Done.")
    except Exception as e:
        print(f"  Skipped: {e}")

# --- Drop vector tables ---
for table in [f"vdb_{VEC_NAMESPACE}", "vdb_entities", "vdb_relationships", "vdb_chunks"]:
    print(f"Dropping table '{table}'...")
    try:
        op = database.update_ddl([f"DROP TABLE IF EXISTS {table}"])
        op.result()
        print("  Done.")
    except Exception as e:
        print(f"  Skipped: {e}")

# --- Clean up temp directory ---
import shutil
if 'work_dir' in dir():
    shutil.rmtree(work_dir, ignore_errors=True)
    print(f"Removed temp dir: {work_dir}")

print("\nCleanup completed!")

---
## References

- [Cloud Spanner Documentation](https://cloud.google.com/spanner/docs)
- [Spanner Graph Overview](https://cloud.google.com/spanner/docs/graph/overview)
- [Spanner Vector Search](https://cloud.google.com/spanner/docs/find-k-nearest-neighbors)
- [Spanner GQL Reference](https://cloud.google.com/spanner/docs/reference/standard-sql/graph-query-statements)